# Equara AI — FinBERT trên Colab
Để mở web, chạy ô [GIAO DIỆN] ngay bên dưới. Các ô phân tích tương tác dùng lại tin trong 10 phút.

## Giao diện web trên Colab
Chạy riêng ô kế tiếp, chọn ZIP khi được hỏi. Không cần Run all.

In [ ]:
# [GIAO DIỆN] Chạy web app của dự án trong Colab và hiện giao diện
%pip install -q "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "httpx>=0.27,<1" "feedparser>=6.0.11,<7"
import json, os, subprocess, sys, time, urllib.request

from pathlib import Path
from zipfile import ZipFile
from tempfile import mkdtemp
from google.colab import files

if not globals().get("PROJECT_DIR") or not (Path(PROJECT_DIR) / "backend/app.py").is_file():
    print("Chọn ZIP dự án.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Cần chọn đúng một ZIP dự án.")
    folder = Path(mkdtemp(dir="/content"))
    with ZipFile(next(iter(uploaded))) as archive:
        archive.extractall(folder)
    projects = list(folder.glob("**/backend/app.py"))
    if len(projects) != 1:
        raise RuntimeError("ZIP phải chứa đúng một dự án có backend/app.py.")
    PROJECT_DIR = str(projects[0].parents[1])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(Path(PROJECT_DIR) / "requirements.txt")], check=True)

USE_HF_HTTP = False #@param {type:"boolean"}
WEB_PORT = 8012 if USE_HF_HTTP else 8000
WEB_BASE = f"http://127.0.0.1:{WEB_PORT}"
web_log_path = os.path.join(PROJECT_DIR, "artifacts", "web_app_colab.log")


def _web_health():
    try:
        with urllib.request.urlopen(WEB_BASE + "/api/health", timeout=3) as response:
            return json.load(response)
    except Exception:
        return None


try:
    from google.colab import userdata as _secrets
    _secret_value = (_secrets.get("EODHD_API_TOKEN") or "").strip()
except Exception:
    _secret_value = ""
print("Secret EODHD_API_TOKEN:", "đọc được" if _secret_value else "chưa đọc được. Mở Secrets và bật Notebook access cho notebook này.")
web_health = _web_health()
if web_health and bool(web_health.get("eodhd_token_configured")) != bool(_secret_value):
    raise RuntimeError("Quyền Secret đã thay đổi. Khởi động lại phiên Colab và chạy lại các ô để web nhận cấu hình mới.")
if not web_health:
    env = dict(os.environ, DEMO_MODE="live", INFERENCE_PROVIDER="transformers-local", LOCAL_INFERENCE_DEVICE="auto")
    for key in ("HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN", "HF_INFERENCE_URL"):
        env.pop(key, None)
    for key in ("EODHD_API_TOKEN", "EODHD_TOKEN"):
        env.pop(key, None)
    if _secret_value:
        env["EODHD_API_TOKEN"] = _secret_value
    if USE_HF_HTTP:
        hf_port = 8013
        hf_url = f"http://127.0.0.1:{hf_port}"
        if not globals().get("web_hf_process") or web_hf_process.poll() is not None:
            with open(os.path.join(PROJECT_DIR, "artifacts", "hf_colab.log"), "w") as hf_log:
                web_hf_process = subprocess.Popen(
                    [sys.executable, "-m", "scripts.serve_hf_api", "--port", str(hf_port)],
                    cwd=PROJECT_DIR, env={k:v for k,v in env.items() if k not in ("EODHD_API_TOKEN","EODHD_TOKEN")}, stdout=hf_log, stderr=subprocess.STDOUT,
                )
        for _ in range(300):
            if web_hf_process.poll() is not None:
                raise RuntimeError("Máy chủ FinBERT đã dừng. Xem artifacts/hf_colab.log.")
            try:
                with urllib.request.urlopen(hf_url + "/health", timeout=2) as response:
                    if json.load(response).get("model") == "ProsusAI/finbert":
                        break
            except OSError:
                pass
            time.sleep(1)
        else:
            raise RuntimeError("Máy chủ FinBERT chưa sẵn sàng.")
        env.update(INFERENCE_PROVIDER="hf-inference", HF_INFERENCE_URL=hf_url + "/models/ProsusAI/finbert")
    web_log = open(web_log_path, "w", encoding="utf-8")
    web_server = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", str(WEB_PORT)],
        cwd=PROJECT_DIR, env=env, stdout=web_log, stderr=subprocess.STDOUT,
    )
    for _ in range(300):  # tối đa 5 phút cho lần nạp model đầu tiên
        web_health = _web_health()
        if web_health:
            break
        if web_server.poll() is not None:
            raise RuntimeError("Web app đã dừng. Xem log: " + web_log_path)
        time.sleep(1)
    else:
        raise RuntimeError("Web app chưa sẵn sàng sau 5 phút. Xem log: " + web_log_path)

print("Web app đang chạy:", {k: web_health.get(k) for k in ("data_mode", "inference_provider", "inference_revision", "inference_device", "eodhd_token_configured")})
try:
    from google.colab import output as colab_output
    colab_output.serve_kernel_port_as_window(WEB_PORT, anchor_text="Mở giao diện web trong tab mới")
    colab_output.serve_kernel_port_as_iframe(WEB_PORT, height=1100)
except ImportError:
    print("Không ở Colab: mở", WEB_BASE)

In [ ]:
%pip install -q "transformers>=4.44,<6" "ipywidgets>=8,<9" "pandas>=2,<4" requests

In [ ]:
import html
import re
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from urllib.parse import urlsplit, urlunsplit

import pandas as pd
import requests
import torch
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display
from transformers import AutoModelForSequenceClassification, AutoTokenizer

try:
    from google.colab import output as colab_output, userdata as colab_userdata
    colab_output.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    colab_userdata = None
    IN_COLAB = False


def get_eodhd_api_token():
    """Read the optional token from Colab Secrets without displaying it."""
    if colab_userdata is None:
        return None
    try:
        return colab_userdata.get("EODHD_API_TOKEN")
    except Exception:
        return None

MODEL_ID = "ProsusAI/finbert"
# Pin to the model snapshot used for local CPU validation in this project.
MODEL_REVISION = "0bc2ac10dda6929bcb1f77ccc43c7ec5c424f031"
NEWS_LOOKBACK_DAYS = 90
REQUEST_TIMEOUT_SECONDS = 25
MAX_STOCK_HEADLINES = 15
MAX_MICRO_PER_PORTFOLIO_SYMBOL = 2
MAX_MACRO_PER_PORTFOLIO = 4
MAX_PORTFOLIO_SYMBOLS = 5
EGYPT_TERMS = ("egypt", "egyptian", "cairo", "egx")

COMPANIES = {
    "COMI": {"name": "Commercial International Bank", "aliases": ("COMI", "CIB", "Commercial International Bank")},
    "ETEL": {"name": "Telecom Egypt", "aliases": ("ETEL", "Telecom Egypt")},
    "FWRY": {"name": "Fawry", "aliases": ("FWRY", "Fawry")},
    "ADIB": {"name": "Abu Dhabi Islamic Bank Egypt", "aliases": ("ADIB", "Abu Dhabi Islamic Bank", "ADIB Egypt")},
    "EFIH": {"name": "E-finance for Digital and Financial Investments", "aliases": ("EFIH", "E-finance", "E-finance for Digital and Financial Investments")},
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))

print(f"Runtime: {'Google Colab' if IN_COLAB else 'Python notebook'}")
print(f"Inference device: {DEVICE} (CPU is supported; GPU is optional)")
_secret_available = bool(get_eodhd_api_token())
print(f"EODHD Secret configured: {_secret_available} (value hidden).")
_secret_available = None
print("News mode: EODHD ticker priority → Google News RSS fallback; BBC macro RSS → Google News RSS fallback.")

In [ ]:
print(f"Loading {MODEL_ID} at revision {MODEL_REVISION} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model.to(DEVICE)
model.eval()

_raw_id2label = model.config.id2label
ID2LABEL = {
    i: str(_raw_id2label.get(i, _raw_id2label.get(str(i), f"LABEL_{i}"))).strip().lower()
    for i in range(model.config.num_labels)
}
LABELS = ("positive", "negative", "neutral")
if set(ID2LABEL.values()) != set(LABELS):
    raise RuntimeError(f"Unexpected labels from {MODEL_ID}: {ID2LABEL}")

print(f"Loaded model: {MODEL_ID}")
print(f"Class labels: {', '.join(ID2LABEL[i] for i in range(len(ID2LABEL)))}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def classify_texts(texts, batch_size=16):
    # Run local FinBERT and return one softmax probability row per input text.
    texts = [str(text).strip() for text in texts]
    if not texts:
        return []
    results = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        encoded = {name: tensor.to(DEVICE) for name, tensor in encoded.items()}
        with torch.inference_mode():
            probabilities = torch.softmax(model(**encoded).logits, dim=-1).cpu()
        for row in probabilities:
            values = {ID2LABEL[i]: float(row[i]) for i in range(len(ID2LABEL))}
            if set(values) != set(LABELS):
                raise RuntimeError(f"Missing class probabilities: {values}")
            if abs(sum(values.values()) - 1.0) > 1e-5:
                raise RuntimeError(f"Softmax values do not sum to 1: {values}")
            results.append(values)
    return results


# Controlled, synthetic inputs check only label mapping and softmax behavior.
# They are kept separate from RSS collection and never count as live news.
LABEL_SMOKE_CASES = [
    ("positive", "The company reported record profits and raised its earnings outlook."),
    ("negative", "The bank warned of widening losses and a sharp decline in revenue."),
    ("neutral", "The board will meet on Tuesday to discuss the annual report."),
]
_smoke_probabilities = classify_texts([text for _, text in LABEL_SMOKE_CASES])
_smoke_rows = []
for (expected, text), scores in zip(LABEL_SMOKE_CASES, _smoke_probabilities):
    _smoke_rows.append({
        "test": "controlled label check",
        "expected_for_check": expected,
        "predicted": max(scores, key=scores.get),
        "text": text,
        **{label: scores[label] for label in LABELS},
        "probability_sum": sum(scores.values()),
    })
display(pd.DataFrame(_smoke_rows))
print("These controlled phrases are not news and are excluded from the live RSS analyses.")

In [ ]:
RSS_SESSION = requests.Session()
RSS_SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; EquaraColabDemo/1.0; +https://colab.research.google.com)"
})
GOOGLE_NEWS_RSS = "https://news.google.com/rss/search"
BBC_MIDDLE_EAST_RSS = "https://feeds.bbci.co.uk/news/world/middle_east/rss.xml"
EODHD_NEWS_ENDPOINT = "https://eodhd.com/api/news"

# Per-runtime caches keep the same ticker/feed from being requested twice across
# Single Stock and Portfolio analyses.
# [HOÀN THIỆN] Tin chỉ được dùng lại trong 10 phút (trước đây giữ suốt phiên Colab); quá hạn thì lấy mới.
NEWS_CACHE_TTL_SECONDS = 10 * 60
VN_TZ = timezone(timedelta(hours=7))
EODHD_STOCK_CACHE = {}
STOCK_NEWS_CACHE = {}
MACRO_NEWS_CACHE = None


def _now_utc():
    return datetime.now(timezone.utc)


def _cache_fresh(entry):
    return entry is not None and (_now_utc() - entry["fetched_at"]).total_seconds() < NEWS_CACHE_TTL_SECONDS


def _cached_rows(entry):
    return [{**row, "status": f"cached:{row['status']}", "cached": True} for row in entry["providers"]]


def vn_time(value):
    if isinstance(value, str):
        value = datetime.fromisoformat(value)
    return value.astimezone(VN_TZ).strftime("%H:%M:%S %d/%m/%Y")


def _local_name(tag):
    return tag.rsplit("}", 1)[-1].lower()


def _child_text(element, wanted):
    for child in list(element):
        if _local_name(child.tag) == wanted.lower():
            return "".join(child.itertext()).strip()
    return ""


def _parse_published(raw):
    if not raw:
        return None
    try:
        value = parsedate_to_datetime(raw)
    except (TypeError, ValueError, OverflowError):
        try:
            value = datetime.fromisoformat(raw.replace("Z", "+00:00"))
        except ValueError:
            return None
    if value.tzinfo is None:
        value = value.replace(tzinfo=timezone.utc)
    return value.astimezone(timezone.utc)


def _rss_items(url, params, kind, symbol=None, fallback_source="RSS"):
    response = RSS_SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    root = ET.fromstring(response.content)
    parsed = []
    cutoff = datetime.now(timezone.utc) - timedelta(days=NEWS_LOOKBACK_DAYS)
    for element in root.iter():
        if _local_name(element.tag) != "item":
            continue
        title = html.unescape(_child_text(element, "title")).strip()
        if not title:
            continue
        description = html.unescape(re.sub(r"<[^>]+>", " ", _child_text(element, "description"))).strip()
        link = _child_text(element, "link").strip() or None
        published = _parse_published(_child_text(element, "pubDate"))
        if published is not None and published < cutoff:
            continue
        source_node = next((child for child in list(element) if _local_name(child.tag) == "source"), None)
        source = "".join(source_node.itertext()).strip() if source_node is not None else fallback_source
        parsed.append({
            "title": title,
            "url": link,
            "published_at": published.isoformat() if published else None,
            "source": source or fallback_source,
            "kind": kind,
            "symbol": symbol,
            "description": description or None,
        })
    return parsed


def _is_egypt_related(text):
    lowered = (text or "").casefold()
    return any(term in lowered for term in EGYPT_TERMS)


def _canonical_url(url):
    if not url:
        return ""
    parts = urlsplit(url.strip())
    return urlunsplit((parts.scheme.lower(), parts.netloc.lower(), parts.path.rstrip("/"), "", ""))


def _normalize_title(title):
    value = re.sub(r"\s+", " ", title.casefold()).strip()
    return re.sub(r"[^\w\s]", "", value)


def deduplicate_and_sort(items, limit):
    seen = set()
    unique = []
    for item in items:
        key = _canonical_url(item.get("url")) or _normalize_title(item["title"])
        if key and key not in seen:
            seen.add(key)
            unique.append(item)
    unique.sort(
        key=lambda item: (1 if item.get("published_at") else 0, item.get("published_at") or ""),
        reverse=True,
    )
    return unique[:limit]


def _eodhd_status_items(symbol):
    if _cache_fresh(EODHD_STOCK_CACHE.get(symbol)):
        entry = EODHD_STOCK_CACHE[symbol]
        return list(entry.get("items", [])), entry.get("status", "cached_unknown")

    api_token = get_eodhd_api_token()
    if not isinstance(api_token, str) or not api_token.strip():
        return [], "skipped_secret_unavailable"

    response = None
    items = []
    status = "error:unknown"
    try:
        response = RSS_SESSION.get(
            EODHD_NEWS_ENDPOINT,
            params={"s": symbol, "limit": MAX_STOCK_HEADLINES, "fmt": "json", "api_token": api_token},
            timeout=REQUEST_TIMEOUT_SECONDS,
            allow_redirects=False,
        )
        if not 200 <= response.status_code < 300:
            status = f"http_error:{response.status_code}"
        else:
            payload = response.json()
            if not isinstance(payload, list):
                status = "error:invalid_response"
            elif not payload:
                status = "http_200_empty:0"
            else:
                cutoff = datetime.now(timezone.utc) - timedelta(days=NEWS_LOOKBACK_DAYS)
                for row in payload:
                    if not isinstance(row, dict) or not str(row.get("title", "")).strip():
                        continue
                    published = _parse_published(str(row.get("date") or ""))
                    if published is not None and published < cutoff:
                        continue
                    items.append({
                        "title": str(row["title"]).strip(),
                        "url": row.get("link"),
                        "published_at": published.isoformat() if published else None,
                        "source": "EODHD",
                        "kind": "micro",
                        "symbol": symbol,
                        "description": str(row.get("content") or "").strip() or None,
                    })
                items = deduplicate_and_sort(items, MAX_STOCK_HEADLINES)
                status = f"http_200:{len(items)}" if items else "http_200_filtered_empty:0"
    except requests.Timeout:
        status = "error:Timeout"
    except requests.RequestException as error:
        status = f"error:{type(error).__name__}"
    except (ValueError, TypeError):
        status = "error:invalid_response"
    finally:
        if response is not None:
            response.close()
        response = None
        api_token = None

    EODHD_STOCK_CACHE[symbol] = {"items": list(items), "status": status, "fetched_at": _now_utc()}
    return items, status


def fetch_google_stock_news(symbol, limit=15):
    company = COMPANIES.get(symbol, {}).get("name", symbol)
    query = f'"{company}" Egypt EGX stock when:{NEWS_LOOKBACK_DAYS}d'
    items = _rss_items(
        GOOGLE_NEWS_RSS,
        {"q": query, "hl": "en-US", "gl": "US", "ceid": "US:en"},
        kind="micro",
        symbol=f"{symbol}.EGX",
        fallback_source="Google News RSS",
    )
    aliases = tuple(alias.casefold() for alias in COMPANIES.get(symbol, {}).get("aliases", (symbol,)))
    items = [
        item for item in items
        if any(alias in f"{item['title']} {item.get('description') or ''}".casefold() for alias in aliases)
    ]
    return deduplicate_and_sort(items, limit)


def fetch_stock_news(symbol, limit=15, force_refresh=False):
    normalized = symbol if symbol.endswith(".EGX") else f"{symbol}.EGX"
    short = normalized.removesuffix(".EGX")
    entry = STOCK_NEWS_CACHE.get(normalized)
    if not force_refresh and _cache_fresh(entry):
        return list(entry["items"])[:limit], _cached_rows(entry)
    if force_refresh:
        EODHD_STOCK_CACHE.pop(normalized, None)

    fetched_at = _now_utc()
    eodhd_items, eodhd_status = _eodhd_status_items(normalized)
    providers = [{"provider": "EODHD", "kind": f"micro:{normalized}", "status": eodhd_status}]
    if eodhd_items:
        items = eodhd_items
        providers.append({"provider": "Google News RSS", "kind": f"micro:{normalized}", "status": "skipped_eodhd_nonempty"})
    else:
        try:
            items = fetch_google_stock_news(short, limit=MAX_STOCK_HEADLINES)
            google_status = f"used:{len(items)}" if items else "empty:0"
        except Exception as error:
            items = []
            google_status = f"error:{type(error).__name__}"
        providers.append({"provider": "Google News RSS", "kind": f"micro:{normalized}", "status": google_status})

    items = deduplicate_and_sort(items, MAX_STOCK_HEADLINES)
    providers = [{**row, "fetched_at": fetched_at.isoformat()} for row in providers]
    STOCK_NEWS_CACHE[normalized] = {"items": list(items), "providers": list(providers), "fetched_at": fetched_at}
    return items[:limit], providers


def fetch_bbc_macro_news(limit=15, force_refresh=False):
    global MACRO_NEWS_CACHE
    if not force_refresh and _cache_fresh(MACRO_NEWS_CACHE):
        return list(MACRO_NEWS_CACHE["items"])[:limit], _cached_rows(MACRO_NEWS_CACHE)

    fetched_at = _now_utc()
    providers = []
    bbc_items = []
    try:
        bbc_items = _rss_items(BBC_MIDDLE_EAST_RSS, None, kind="macro", fallback_source="BBC RSS")
        bbc_items = [
            item for item in bbc_items
            if _is_egypt_related(f"{item['title']} {item.get('description') or ''}")
        ]
        bbc_items = deduplicate_and_sort(bbc_items, MAX_STOCK_HEADLINES)
        providers.append({"provider": "BBC RSS", "kind": "macro", "status": f"used:{len(bbc_items)}" if bbc_items else "empty:0"})
    except Exception as error:
        providers.append({"provider": "BBC RSS", "kind": "macro", "status": f"error:{type(error).__name__}"})

    if bbc_items:
        items = bbc_items
        providers.append({"provider": "Google News RSS", "kind": "macro", "status": "skipped_bbc_nonempty"})
    else:
        try:
            items = _rss_items(
                GOOGLE_NEWS_RSS,
                {"q": f"Egypt economy OR Egyptian economy OR EGX when:{NEWS_LOOKBACK_DAYS}d", "hl": "en-US", "gl": "US", "ceid": "US:en"},
                kind="macro",
                fallback_source="Google News RSS",
            )
            items = [item for item in items if _is_egypt_related(f"{item['title']} {item.get('description') or ''}")]
            items = deduplicate_and_sort(items, MAX_STOCK_HEADLINES)
            providers.append({"provider": "Google News RSS", "kind": "macro", "status": f"used:{len(items)}" if items else "empty:0"})
        except Exception as error:
            items = []
            providers.append({"provider": "Google News RSS", "kind": "macro", "status": f"error:{type(error).__name__}"})

    providers = [{**row, "fetched_at": fetched_at.isoformat()} for row in providers]
    MACRO_NEWS_CACHE = {"items": list(items), "providers": list(providers), "fetched_at": fetched_at}
    return items[:limit], providers


def normalize_symbols(raw_symbols):
    symbols = []
    for value in raw_symbols:
        symbol = str(value).strip().upper()
        if symbol.endswith(".EGX"):
            symbol = symbol[:-4]
        if not re.fullmatch(r"[A-Z0-9]{2,8}", symbol):
            raise ValueError(f"Invalid EGX symbol: {value!r}")
        symbol = f"{symbol}.EGX"
        if symbol in symbols:
            raise ValueError(f"Duplicate symbol: {symbol}")
        symbols.append(symbol)
    if not 1 <= len(symbols) <= MAX_PORTFOLIO_SYMBOLS:
        raise ValueError(f"Choose between 1 and {MAX_PORTFOLIO_SYMBOLS} distinct symbols.")
    return symbols


In [ ]:
def aggregate_probabilities(probability_rows):
    if not probability_rows:
        raise ValueError("There are no live headlines to analyze.")
    means = {
        label: sum(row[label] for row in probability_rows) / len(probability_rows)
        for label in LABELS
    }
    positive, negative, neutral = (means[label] for label in LABELS)
    if positive > negative and positive > neutral:
        sentiment = "Bullish"
    elif negative > positive and negative > neutral:
        sentiment = "Bearish"
    else:
        sentiment = "Neutral"
    return means, sentiment, positive - negative


def _attach_inference(headlines):
    if not headlines:
        raise ValueError("No real EODHD or RSS headlines were returned; no inference was run.")
    probabilities = classify_texts([item["title"] for item in headlines])
    output = []
    for item, scores in zip(headlines, probabilities):
        output.append({**item, "probabilities": scores, "predicted_label": max(scores, key=scores.get)})
    means, sentiment, score = aggregate_probabilities(probabilities)
    return {
        "analysis_status": "completed",
        "data_mode": "live_news_local_transformers",
        "model": MODEL_ID,
        "revision": MODEL_REVISION,
        "device": str(DEVICE),
        "sentiment": sentiment,
        "probabilities": means,
        "score": score,
        "news_count": len(output),
        "headlines": output,
    }


def analyze_stock(symbol, force_refresh=False):
    normalized = normalize_symbols([symbol])[0]
    micro, micro_status = fetch_stock_news(normalized, limit=MAX_STOCK_HEADLINES, force_refresh=force_refresh)
    macro, macro_status = fetch_bbc_macro_news(limit=MAX_STOCK_HEADLINES, force_refresh=force_refresh)
    headlines = deduplicate_and_sort(micro + macro, MAX_STOCK_HEADLINES)
    result = _attach_inference(headlines)
    result["scope"] = "stock"
    result["symbols"] = [normalized]
    result["providers"] = micro_status + macro_status
    result["warnings"] = []
    if len(headlines) < 10:
        result["warnings"].append(f"Only {len(headlines)} recent EODHD/RSS headlines were found; target is about 10–15.")
    return result


def analyze_portfolio(raw_symbols, force_refresh=False):
    symbols = normalize_symbols(raw_symbols)
    collected = []
    providers = []
    warnings = []
    for normalized in symbols:
        micro, source_status = fetch_stock_news(normalized, limit=MAX_MICRO_PER_PORTFOLIO_SYMBOL, force_refresh=force_refresh)
        collected.extend(micro)
        providers.extend(source_status)
        if not micro:
            warnings.append(f"No recent company-specific EODHD/RSS headline for {normalized}.")
    macro, macro_status = fetch_bbc_macro_news(limit=MAX_MACRO_PER_PORTFOLIO, force_refresh=force_refresh)
    collected.extend(macro)
    providers.extend(macro_status)
    max_items = MAX_PORTFOLIO_SYMBOLS * MAX_MICRO_PER_PORTFOLIO_SYMBOL + MAX_MACRO_PER_PORTFOLIO
    headlines = deduplicate_and_sort(collected, max_items)
    result = _attach_inference(headlines)
    result["scope"] = "portfolio"
    result["symbols"] = symbols
    result["providers"] = providers
    result["warnings"] = warnings
    if len(macro) < MAX_MACRO_PER_PORTFOLIO:
        result["warnings"].append(f"Only {len(macro)} recent macro headlines were found; target is {MAX_MACRO_PER_PORTFOLIO}.")
    return result


def display_freshness(providers):
    # [HOÀN THIỆN] Hiện rõ tin vừa lấy trực tiếp hay đang dùng lại, lấy lúc mấy giờ (giờ Việt Nam).
    rows = [row for row in providers if row.get("fetched_at")]
    fresh = [row for row in rows if not row.get("cached")]
    reused = [row for row in rows if row.get("cached")]
    if fresh:
        when = vn_time(min(row["fetched_at"] for row in fresh))
        color, title = ("#e5f3ec", "#0f5442"), f"● Tin lấy trực tiếp từ internet lúc {when} (giờ Việt Nam)"
    else:
        oldest = min((row["fetched_at"] for row in reused), default=None)
        minutes = max(0, int((_now_utc() - datetime.fromisoformat(oldest)).total_seconds() // 60)) if oldest else 0
        color, title = ("#fff6e3", "#7a5410"), f"↺ Dùng lại tin đã lấy lúc {vn_time(oldest) if oldest else '?'} ({minutes} phút trước)"
    fresh_groups = {row["kind"] for row in fresh}
    reused_groups = {row["kind"] for row in reused} - fresh_groups
    note = (f"{len(fresh_groups)} nhóm tin vừa lấy mới, {len(reused_groups)} nhóm dùng lại "
            f"(mỗi mã là 1 nhóm, tin vĩ mô Ai Cập là 1 nhóm). "
            f"Tin chỉ được dùng lại trong {NEWS_CACHE_TTL_SECONDS // 60} phút; tick ô \"Lấy tin mới\" để lấy lại ngay.")
    display(HTML(
        f'<div style="background:{color[0]};color:{color[1]};border-radius:10px;padding:10px 14px;margin:6px 0;font-family:sans-serif">'
        f'<b style="font-size:15px">{html.escape(title)}</b><br><span style="font-size:12px">{html.escape(note)}</span></div>'
    ))


def display_analysis(result):
    display_freshness(result["providers"])
    print(
        f"{result['scope'].upper()} | {', '.join(result['symbols'])} | "
        f"{result['sentiment']} | {result['news_count']} headlines | "
        f"{result['model']} local on {result['device']}"
    )
    summary = pd.DataFrame([{
        "sentiment": result["sentiment"],
        "score (mean positive - negative)": result["score"],
        **result["probabilities"],
    }])
    display(summary)
    provider_counts = {"EODHD": 0, "Google News RSS": 0, "BBC RSS": 0}
    for item in result["headlines"]:
        source = item.get("source", "")
        provider = item.get("provider") or (source if source in {"EODHD", "BBC RSS"} else "Google News RSS")
        provider_counts[provider] = provider_counts.get(provider, 0) + 1
    display(pd.DataFrame([{"provider": name, "headlines_in_analysis": count} for name, count in provider_counts.items()]))
    display(pd.DataFrame([
        {**{k: v for k, v in row.items() if k != "fetched_at"},
         "fetched_vn": vn_time(row["fetched_at"]) if row.get("fetched_at") else None}
        for row in result["providers"]
    ]))
    columns = [
        "published_at", "published_vn", "provider", "source", "kind", "symbol", "title",
        "predicted_label", "positive", "negative", "neutral", "url",
    ]
    rows = []
    for item in result["headlines"]:
        source = item.get("source", "")
        provider = item.get("provider") or (source if source in {"EODHD", "BBC RSS"} else "Google News RSS")
        rows.append({
            **{key: item.get(key) for key in columns if key not in {"positive", "negative", "neutral", "provider", "published_vn"}},
            "published_vn": vn_time(item["published_at"]) if item.get("published_at") else "không rõ",
            "provider": provider,
            **item["probabilities"],
        })
    display(pd.DataFrame(rows, columns=columns))
    if result["warnings"]:
        for warning in result["warnings"]:
            display(HTML(f'<div style="color:#9a6700"><b>Data limitation:</b> {html.escape(warning)}</div>'))
    print(f"Mode: live EODHD-priority/RSS-fallback news (dùng lại tối đa {NEWS_CACHE_TTL_SECONDS // 60} phút) + local Transformers; no fixture scores and no HF Inference API call.")


## Phân tích tương tác

Nhập một mã hoặc một danh sách tối đa 5 mã EGX. Khi bấm nút, notebook tải RSS trực tiếp, lọc trong 90 ngày gần nhất khi ngày phát hành có sẵn, khử trùng, chạy local FinBERT theo batch và hiển thị toàn bộ tiêu đề cùng ba xác suất.

Mỗi lần bấm nút sẽ phát sinh các request RSS tương ứng; inference chạy trong runtime. Khung màu xanh đầu kết quả ghi giờ lấy tin (giờ Việt Nam); khung màu vàng nghĩa là đang dùng lại tin đã lấy trong 10 phút gần nhất. Tick ô "Lấy tin mới" để gọi lại nguồn tin ngay. Notebook không gửi request inference tính phí tới Hugging Face.

In [ ]:
stock_input = widgets.Text(
    value="COMI",
    description="One stock:",
    placeholder="COMI",
    layout=widgets.Layout(width="420px"),
)
stock_button = widgets.Button(description="Analyze stock", button_style="primary")
stock_output = widgets.Output()
# [HOÀN THIỆN] Tick để bỏ qua tin đã lấy, gọi lại nguồn tin ngay lúc bấm.
refresh_box = widgets.Checkbox(value=False, description="Lấy tin mới (bỏ qua tin đã lấy)", indent=False)


def _run_stock(_):
    with stock_output:
        clear_output(wait=True)
        try:
            print(f"Input: {stock_input.value}")
            result = analyze_stock(stock_input.value, force_refresh=refresh_box.value)
            display_analysis(result)
        except Exception as error:
            print(f"Analysis failed: {type(error).__name__}: {error}")


stock_button.on_click(_run_stock)
display(widgets.VBox([refresh_box, stock_input, stock_button, stock_output]))

portfolio_input = widgets.Text(
    value="COMI, ETEL, FWRY, ADIB, EFIH",
    description="Portfolio:",
    placeholder="COMI, ETEL, FWRY",
    layout=widgets.Layout(width="620px"),
)
portfolio_button = widgets.Button(description="Analyze portfolio (max 5)", button_style="success")
portfolio_output = widgets.Output()


def _run_portfolio(_):
    with portfolio_output:
        clear_output(wait=True)
        try:
            raw = [value.strip() for value in portfolio_input.value.split(",") if value.strip()]
            print(f"Input: {raw}")
            result = analyze_portfolio(raw, force_refresh=refresh_box.value)
            display_analysis(result)
        except Exception as error:
            print(f"Analysis failed: {type(error).__name__}: {error}")


portfolio_button.on_click(_run_portfolio)
display(widgets.VBox([portfolio_input, portfolio_button, portfolio_output]))

## Cách đọc kết quả và giới hạn

- Bảng đếm nguồn hiển thị số headline thực sự đưa vào từng lần phân tích theo provider; bảng trạng thái ghi kết quả EODHD/RSS, gồm `empty:0`, lỗi và provider được bỏ qua. Bảng headline hiển thị provider, publisher/source, ngày, tiêu đề, nhãn và ba xác suất Positive/Negative/Neutral.
- Sentiment tổng hợp giữ nguyên: trung bình số học không trọng số của ba xác suất. Bullish/Bearish chỉ khi xác suất tương ứng là cực đại duy nhất; trường hợp còn lại là Neutral. Score = mean(positive) − mean(negative). Đây là công thức tái dựng vì bài báo không công bố chi tiết.
- Một mã dùng tối đa 15 tiêu đề micro + macro. Danh mục tối đa 5 mã dùng tối đa 2 tiêu đề micro mỗi mã + 4 macro, tổng tối đa 14.
- Token chỉ được đọc từ Colab Secrets bằng `google.colab.userdata`; giá trị không được in. Cache theo runtime tránh request EODHD trùng lặp trong Single Stock và Portfolio.

### Kết quả Colab đã xác minh — 01-11-2026

| Phân tích | EODHD | Google News RSS | BBC RSS | Headline dùng | Positive | Negative | Neutral | Score | Sentiment |
|---|---|---:|---:|---:|---:|---:|---:|---:|---|
| COMI.EGX | 0 (kết quả preflight HTTP 200/0 do người dùng cung cấp, tái sử dụng) | 9 | 2 | 11 | 0.273267 | 0.179503 | 0.547230 | 0.093763 | Neutral |
| COMI, ETEL, FWRY, ADIB, EFIH | 0; bốn request mới đều HTTP 200/0 | 6 | 2 | 8 | 0.181245 | 0.243448 | 0.575306 | -0.062203 | Neutral |

Phân bổ headline micro dùng trong Portfolio: COMI 2, ETEL 2, FWRY 0, ADIB 1, EFIH 1; macro BBC RSS 2. Google News fallback cho FWRY rỗng. Không có EODHD headline thực tế trong hai kết quả; các tiêu đề là Google News RSS và BBC RSS. Single Stock tái sử dụng kết quả COMI, không gửi EODHD request. Portfolio chỉ gửi thêm bốn request, mỗi mã ETEL/FWRY/ADIB/EFIH đúng một lần; không gọi general/macro EODHD. Tổng số headline thấp hơn mục tiêu 10–15 ở Single Stock và giới hạn 14 ở Portfolio. BBC có thể chứa tin xã hội/chính trị không phản ánh riêng biến động cổ phiếu.

EODHD trả HTTP 200 nhưng `NEWS_COUNT=0` cho COMI và bốn mã Portfolio. Điều này xác nhận request được chấp nhận ở endpoint nhưng không chứng minh có tin cho các mã đó. Tích hợp EODHD-first và fallback đã chạy; dữ liệu EODHD thực tế thu được trong lượt kiểm chứng này là **0 headline**. Không có bằng chứng để kết luận nguyên nhân là quyền gói hay thiếu tin trong khoảng thời gian trả về; không nâng cấp gói hoặc phát sinh thanh toán.

FinBERT local tiếp tục dùng `ProsusAI/finbert`, revision `0bc2ac10dda6929bcb1f77ccc43c7ec5c424f031`, Transformers/PyTorch trên CPU, giữ nguyên tokenization, truncation, batching và softmax đã kiểm thử. Hai phân tích không gọi HF Inference API và không dùng fixture.

### Kiểm tra mapping nhãn

Các câu dưới đây là input kiểm soát tổng hợp, chỉ để kiểm tra mapping và softmax; chúng không được đưa vào phân tích tin RSS.

| Câu kiểm soát | Positive | Negative | Neutral | Nhãn lớn nhất |
|---|---:|---:|---:|---|
| Company reported record profits and raised its earnings outlook | 0.955702 | 0.018400 | 0.025898 | Positive |
| Bank warned of widening losses and a sharp decline in revenue | 0.010671 | 0.973866 | 0.015463 | Negative |
| Board will meet on Tuesday to discuss the annual report | 0.022532 | 0.045032 | 0.932435 | Neutral |

## Giới hạn

- Headline RSS có thể ít, cũ, trùng hoặc không chuyên về thị trường; ngày quá 90 ngày bị loại.
- Tin của mỗi ticker và tin vĩ mô được dùng lại tối đa 10 phút để Single Stock và Portfolio không gọi lặp; quá 10 phút hoặc tick "Lấy tin mới" thì gọi lại nguồn tin. RSS được dùng khi EODHD trả 0 headline hoặc lỗi.
- Sentiment headline là kết quả phân loại ngôn ngữ, không phải khuyến nghị đầu tư.
- Colab có thể reset runtime; khi đó cache và model đã nạp sẽ mất, còn model weights có thể cần tải lại.


## [Nâng cấp C] Danh mục theo vốn: độ phủ tin

Phần này dùng đúng code của tab "Danh mục theo vốn" trên web (`backend/upgrades/coverage*.py`, `relevance.py`), notebook không chép lại công thức.

- **Lọc đúng doanh nghiệp:** tên công ty phải có trong tiêu đề (so nguyên từ, nên "COMI" không khớp "coming"); loại tin thể thao và trang báo giá, biểu đồ; tên "ADIB" hay "Abu Dhabi Islamic Bank" phải đi kèm chữ chỉ Ai Cập, vì đó cũng là tên ngân hàng mẹ ở UAE. Tin bị loại được liệt kê kèm lý do.
- **Điểm mã** s = trung bình P+ − trung bình P− trên tin riêng của mã, đúng công thức của bản tái dựng.
- **Độ phủ tin** C = phần vốn của các mã có ít nhất 1 tin riêng phù hợp. Ví dụ A 50% có tin, B 30% có tin, C 20% không tìm thấy tin: độ phủ 80%. Mã nào có dưới 10 tin phù hợp thì gọi thêm truy vấn Google rộng hơn (bậc 2), nên "không tìm thấy" nghĩa là cả hai truy vấn đều không thấy.
- **Đóng góp** = (vốn của mã / độ phủ) × điểm mã. Cộng các đóng góp ra điểm chung S, tính trên phần vốn có tin. Ví dụ trên với A +0,6 và B −0,2: S = (50 × 0,6 + 30 × (−0,2)) / 80 = +0,30.
- Chỉ phân tích các tin đã chấm; mã chưa có dữ liệu không được gán điểm thay thế.
- **Tin kinh tế Ai Cập** cho một điểm riêng, không cộng vào S. Chỉ giữ tiêu đề có từ về kinh tế hay thị trường (lạm phát, lãi suất, IMF, chứng khoán, …); tin chính trị bị loại kèm lý do. Bên cạnh vẫn có điểm theo cách bài báo, tính từ cùng lượt lấy tin.

Kết quả chỉ tóm tắt sắc thái tiêu đề tin tiếng Anh; không phải tư vấn đầu tư và không dự báo giá.

In [ ]:
# [NÂNG CẤP C] Nạp dự án và phần C: độ phủ tin theo vốn (dùng đúng code của web app, không chép lại công thức)
%pip install -q feedparser "matplotlib>=3.8,<4"
import asyncio, glob, os, sys, threading, zipfile


def _find_project():
    hits = glob.glob("/content/**/backend/upgrades/coverage.py", recursive=True)
    return os.path.dirname(os.path.dirname(os.path.dirname(hits[0]))) if hits else None


if not globals().get("PROJECT_DIR") or not os.path.isfile(os.path.join(PROJECT_DIR, "backend", "upgrades", "coverage.py")):
    PROJECT_DIR = _find_project()
if PROJECT_DIR is None:
    from google.colab import files
    print("Chọn ZIP dự án.")
    for name in files.upload():
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall("/content/equara_project")
    PROJECT_DIR = _find_project()
if PROJECT_DIR is None:
    raise RuntimeError("Không thấy backend/upgrades/coverage.py trong file đã tải lên.")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import httpx
from backend.config import Settings
from backend.schemas import normalize_symbol
from backend.services.sentiment import SentimentService
from backend.upgrades.coverage import CoverageConfig
from backend.upgrades.coverage_service import CoverageService
from backend.upgrades.service import UpgradedPortfolioService


class NotebookFinBERT:
    """FinBERT đã nạp ở các ô trên (cùng model, cùng revision), dùng cho phần C. Không nạp model lần hai."""

    provider_name = "colab-notebook"
    request_mode = "FinBERT chạy trong notebook, theo lô"
    revision = MODEL_REVISION
    device_name = str(DEVICE)
    batch_size = 16

    async def classify_many(self, texts):
        return await asyncio.to_thread(classify_texts, list(texts))


# Phần C dùng code bất đồng bộ của web app; chạy nó trên một vòng lặp riêng để nút bấm của notebook gọi được.
_C_LOOP = asyncio.new_event_loop()
threading.Thread(target=_C_LOOP.run_forever, daemon=True).start()


def _run_c(coro, timeout=600):
    return asyncio.run_coroutine_threadsafe(coro, _C_LOOP).result(timeout)


async def _build_coverage():
    settings = Settings(
        demo_mode="live",
        inference_provider="colab-notebook",
        local_model_revision=MODEL_REVISION,
        local_inference_device=str(DEVICE),
        hf_token=None,
        eodhd_api_token=get_eodhd_api_token() or None,  # đọc từ Colab Secrets, không in ra
    )
    client = httpx.AsyncClient(timeout=httpx.Timeout(20.0, connect=8.0), follow_redirects=True)
    base = SentimentService(client, settings)
    base.inference = NotebookFinBERT()
    return CoverageService(UpgradedPortfolioService(base), CoverageConfig.from_environment())


COVERAGE_SERVICE = _run_c(_build_coverage())


def parse_positions(text):
    """'COMI:50, ETEL:30, FWRY:20' → [('COMI.EGX', 50.0), ...]"""
    rows = []
    for chunk in str(text).split(","):
        if chunk.strip():
            symbol, _, weight = chunk.partition(":")
            rows.append((normalize_symbol(symbol), float(weight) if weight.strip() else 1.0))
    if not 1 <= len(rows) <= 5 or len({s for s, _ in rows}) != len(rows) or sum(w for _, w in rows) <= 0:
        raise ValueError("Nhập 1–5 mã khác nhau, dạng MÃ:VỐN, tổng vốn > 0.")
    return rows


def run_coverage(text, force_refresh=False):
    if force_refresh:
        COVERAGE_SERVICE.upgraded.news_cache.clear()
    return _run_c(COVERAGE_SERVICE.analyze(parse_positions(text)))


def _pct(x, digits=1):
    return "—" if x is None else f"{x:.{digits}%}"


def _num(x, digits=3):
    return "—" if x is None else f"{x:+.{digits}f}"


def _day(iso):
    return iso[:10] if iso else "không rõ"


def _counts(counts):
    return "—" if not counts else f"{counts['Bullish']} · {counts['Neutral']} · {counts['Bearish']}"


def display_coverage(result):
    p = result["portfolio"]
    calls = [row for row in result["providers"] if row.get("fetched_at")]
    if calls:
        reused = sum(1 for row in calls if row.get("cached"))
        print(f"Lấy tin lúc {min(row['fetched_at'] for row in calls)[:19]} UTC · {len(calls)} lượt gọi nguồn tin, {reused} lượt dùng lại tin trong 10 phút")
    print(f"Vốn được đại diện: {_pct(p['coverage'], 0)} vốn")
    print(f"Điểm tin doanh nghiệp, tính trên {_pct(p['coverage'], 0)} vốn có tin: {_num(p['score'])}")
    print(p["direction_text"])
    print(p["scope_note"])
    display(pd.DataFrame([{
        "Mã": h["short"], "Vốn": _pct(h["w"]), "Số tin riêng": h["n"], "Tin theo nhãn (+ · 0 · −)": _counts(h["label_counts"]),
        "Ngày đăng": f"{_day(h['oldest_published_at'])} – {_day(h['newest_published_at'])}" if h["has_news"] else "",
        "Điểm mã": _num(h["score"]), "Nhãn": h["label"] if h["has_news"] else "Không tìm thấy tin riêng",
        "Tỷ trọng tính lại": _pct(h["r"]), "Đóng góp": _num(h["contribution"]),
        "Tin bị loại": len(h["rejected"]),
    } for h in result["holdings"]]))
    labels = {"Bullish": "Tin riêng Bullish", "Neutral": "Tin riêng Neutral", "Bearish": "Tin riêng Bearish", "no_news": "Không tìm thấy tin riêng"}
    display(pd.DataFrame([{"Nhóm": labels[k], "Phần vốn": _pct(v)} for k, v in p["capital_by_label"].items()]))
    if p.get("capital_by_label_note"):
        print(p["capital_by_label_note"])
    m, a = result["macro"], result["article_method"]
    display(pd.DataFrame([
        {"Ô": "Tin kinh tế và thị trường Ai Cập (không tính vào điểm trên)", "Điểm": _num(m["score"]) if m else "—", "Nhãn": (m["label"] or "—") if m else "—", "Số tiêu đề": m["n"] if m else 0},
        {"Ô": "Cách bài báo (gộp chung tin vĩ mô và tin doanh nghiệp)", "Điểm": _num(a["score"]) if a else "—", "Nhãn": a["sentiment"] if a else "—", "Số tiêu đề": a["headline_count"] if a else 0},
    ]))
    display(pd.DataFrame([{
        "Mã": h["short"], "Tiêu đề": row["title"], "Đăng": _day(row["published_at"]), "Nguồn": row["source"],
        "P+": round(row["probabilities"]["positive"], 3), "P−": round(row["probabilities"]["negative"], 3), "P0": round(row["probabilities"]["neutral"], 3),
        "s": round(row["score"], 3), "Đóng góp": None if row["contribution"] is None else round(row["contribution"], 4), "Link": row["url"],
    } for h in result["holdings"] for row in h["headlines"]]))
    rejected = [{"Mã": h["short"], "Tiêu đề": row["title"], "Lý do": row["reason"], "Bộ lọc của bản tái dựng": "có giữ" if row["old_filter_keep"] else "cũng loại"}
                for h in result["holdings"] for row in h["rejected"]]
    if rejected:
        print("Tin bị loại và lý do:")
        display(pd.DataFrame(rejected))
    for h in result["holdings"]:
        if not h["has_news"]:
            tried = ", ".join(f"{row['provider']} bậc {row.get('tier', 1)}: {row['status']}" for row in h["providers"])
            print(f"{h['short']}: không tìm thấy tin riêng ({h['candidates_total']} ứng viên). Truy vấn đã gọi: {tried}")
    if m and m["rejected"]:
        print("Tin vĩ mô bị loại. " + m["keep_rule"])
        display(pd.DataFrame([{"Tiêu đề": row["title"], "Lý do": row["reason"], "Nguồn": row["source"]} for row in m["rejected"]]))
    print(result["disclaimer"])


print("Đã nạp phần C từ", PROJECT_DIR, "· tùy chọn thử nghiệm đang bật:", COVERAGE_SERVICE.config.experimental_options_on() or "không có")

In [ ]:
# [NÂNG CẤP C] Nhập danh mục dạng MÃ:VỐN, cách nhau bằng dấu phẩy
# Ô "Lấy tin mới" ở phần Phân tích tương tác cũng áp dụng cho nút này.
coverage_input = widgets.Text(
    value="COMI:50, ETEL:30, FWRY:20",
    description="Danh mục:",
    layout=widgets.Layout(width="620px"),
)
coverage_button = widgets.Button(description="Xem độ phủ tin", button_style="warning")
coverage_output = widgets.Output()


def _run_coverage(_):
    with coverage_output:
        clear_output(wait=True)
        try:
            refresh = globals().get("refresh_box")
            display_coverage(run_coverage(coverage_input.value, force_refresh=bool(refresh and refresh.value)))
        except Exception as error:
            print(f"Phân tích thất bại: {type(error).__name__}: {error}")


coverage_button.on_click(_run_coverage)
display(widgets.VBox([coverage_input, coverage_button, coverage_output]))

## [Đo, tùy chọn] Thu ứng viên tin cho phép đo về sau

Ô dưới gọi `scripts/capture_candidates.py`: lưu mọi tin tìm được cho COMI, ETEL, FWRY, ADIB, EFIH (cả tin bị loại, cả truy vấn Google bậc 2), kèm xác suất FinBERT, giờ lấy và địa chỉ truy vấn (đã bỏ khóa API). Dùng cho ba việc đo tiếp theo: gán nhãn "đúng doanh nghiệp", đếm số tin và độ phủ theo ngày, gán nhãn sắc thái tin EGX. Không cần cho buổi demo.

In [ ]:
# [ĐO, tùy chọn] Thu mọi ứng viên tin của 5 mã ngay lúc này, chấm FinBERT, nén và tải về
import shutil, subprocess

_env = {k: v for k, v in os.environ.items() if k not in ("HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN", "HF_INFERENCE_URL", "EODHD_API_TOKEN", "EODHD_TOKEN")}
_env.update(DEMO_MODE="live", INFERENCE_PROVIDER="transformers-local", LOCAL_INFERENCE_DEVICE="auto")
_token = get_eodhd_api_token()
if _token:
    _env["EODHD_API_TOKEN"] = _token  # chỉ truyền cho tiến trình con; không in ra
_token = None
run = subprocess.run([sys.executable, "-m", "scripts.capture_candidates", "--tier2", "--score"], cwd=PROJECT_DIR, env=_env, capture_output=True, text=True)
_env = None
print(run.stdout[-3000:])
if run.returncode != 0:
    print(run.stderr[-3000:])
else:
    archive = shutil.make_archive("/content/captures", "zip", os.path.join(PROJECT_DIR, "captures"))
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
    print("Đã tạo", archive)

## [Phương án cũ 01/11] Xuất 3 biểu đồ PNG + bảng số liệu

Giữ để tái lập đúng bảng đã chia sẻ ngày 01/11 (giảm theo tuổi tin, trọng số độ rõ, β). Phương án đã chốt là phần C ở trên; không dùng các số này làm kết quả của phần C.


Ô dưới chạy `scripts/run_upgrade_demo.py` với đúng model `ProsusAI/finbert` revision `0bc2ac10…` (giống máy đã kiểm chứng), đối chiếu lại 7 tiêu đề tham chiếu, rồi tải về file zip gồm 3 PNG, `BANG_SO_LIEU.md` và `upgrade_results.json`.

In [ ]:
# [NÂNG CẤP] Chạy script, nén kết quả và tải về
import shutil, subprocess
run = subprocess.run([sys.executable, "-m", "scripts.run_upgrade_demo"], cwd=PROJECT_DIR, capture_output=True, text=True)
print(run.stdout[-4000:])
if run.returncode != 0:
    print(run.stderr[-4000:])
else:
    archive = shutil.make_archive("/content/upgrade_results", "zip", os.path.join(PROJECT_DIR, "artifacts", "upgrade"))
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
    print("Đã tạo", archive)

## [Nâng cấp] Biểu đồ 4: FinBERT có thật sự đọc tin tài chính đúng hơn model thông thường?

Bài báo mục 1 khẳng định model sentiment thông thường (học từ review phim) đọc sai tin tài chính nhưng không đưa con số. Ô dưới chạy `scripts/evaluate_finbert.py`:

- Dữ liệu: tập validation của `zeroshot/twitter-financial-news-sentiment` (tin tài chính tiếng Anh, nhãn Bullish / Bearish / Neutral có sẵn). FinBERT chưa từng học bộ này. Không dùng Financial PhraseBank của README vì FinBERT đã học trên chính bộ đó.
- So FinBERT với `distilbert-base-uncased-finetuned-sst-2-english` (học từ review phim).
- Xuất `chart4_finbert_vs_general.png`, `chart5_confidence_check.png`, `BANG_DANH_GIA.md`, `evaluation_results.json`, `predictions.csv` (đúng các cột của `predict.py` trong README), rồi tải về `evaluation_results.zip`. Mất khoảng 10 phút trên CPU.

In [ ]:
# [NÂNG CẤP] Biểu đồ 4: đo độ chính xác FinBERT vs model review phim, nén và tải về
import shutil, subprocess
run = subprocess.run([sys.executable, "-m", "scripts.evaluate_finbert"], cwd=PROJECT_DIR, capture_output=True, text=True)
print(run.stdout[-4000:])
if run.returncode != 0:
    print(run.stderr[-4000:])
else:
    archive = shutil.make_archive("/content/evaluation_results", "zip", os.path.join(PROJECT_DIR, "artifacts", "evaluation"))
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
    print("Đã tạo", archive)

## [Kiểm tra] Chạy đường gọi Hugging Face API của bài báo, miễn phí (máy chủ tự dựng)

Bài báo (mục 5 và 8) gọi FinBERT qua Hugging Face Inference API. Từ tháng 10/2026 tài khoản Hugging Face miễn phí không còn credit Inference Providers, nên gọi máy chủ Hugging Face bị **HTTP 402**. Mục 8 của bài báo đã đặt sẵn phương án thay thế: tự chạy model (self-hosted).

Ô dưới làm đúng phương án đó, **không cần token, không tốn tiền**:

1. Bật `scripts/serve_hf_api.py`: FinBERT bản ghim `0bc2ac10` (tải miễn phí từ Hugging Face Hub), nhận và trả JSON giống hệt Hugging Face, tại `http://127.0.0.1:8001`.
2. Chạy `scripts/check_hf_api.py` với đúng adapter `backend/providers/hf_inference.py` của bài báo, chỉ đổi địa chỉ gọi. 7 tiêu đề tham chiếu đi qua HTTP thật, được so với xác suất FinBERT local đã lưu, rồi đo gửi theo lô so với gửi từng tiêu đề (3 vòng, lấy trung vị).
3. Nén và tải về `self_hosted_api_check.zip`.

Lần đầu ô này mất 1 đến 2 phút để máy chủ nạp model. Đây là máy chủ tự dựng, không phải máy chủ của Hugging Face, kết quả ghi rõ điều đó.


In [ ]:
# [KIỂM TRA] Máy chủ FinBERT tự dựng theo định dạng Hugging Face: chạy adapter của bài báo qua HTTP thật
%pip install -q "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "httpx>=0.27,<1"
import json, os, shutil, subprocess, sys, time, urllib.request

if not globals().get("PROJECT_DIR"):
    raise RuntimeError("Hãy chạy ô '[NÂNG CẤP C] Nạp dự án và phần C' trước.")

PORT = 8001
BASE = f"http://127.0.0.1:{PORT}"
ENDPOINT = f"{BASE}/models/ProsusAI/finbert"
result_dir = os.path.join(PROJECT_DIR, "artifacts", "hf_api_check")
os.makedirs(result_dir, exist_ok=True)
server_log_path = os.path.join(result_dir, "self_hosted_server.log")


def _get(path):
    try:
        with urllib.request.urlopen(BASE + path, timeout=3) as response:
            return json.load(response)
    except Exception:
        return None


health = _get("/health")
if not health or health.get("status") != "ok":
    server_log = open(server_log_path, "w", encoding="utf-8")
    hf_server = subprocess.Popen(
        [sys.executable, "-m", "scripts.serve_hf_api", "--port", str(PORT)],
        cwd=PROJECT_DIR, stdout=server_log, stderr=subprocess.STDOUT,
    )
    for _ in range(300):  # tối đa 5 phút cho lần nạp model đầu tiên
        health = _get("/health")
        if health and health.get("status") == "ok":
            break
        if hf_server.poll() is not None:
            raise RuntimeError("Máy chủ đã dừng. Xem log: " + server_log_path)
        time.sleep(1)
    else:
        raise RuntimeError("Máy chủ chưa sẵn sàng sau 5 phút. Xem log: " + server_log_path)
print("Máy chủ tự dựng:", health)

# Không đưa HF_TOKEN vào: máy chủ tự dựng không cần, và adapter chỉ gửi token tới huggingface.co
env = {k: v for k, v in os.environ.items() if k not in ("HF_TOKEN", "HUGGINGFACEHUB_API_TOKEN")}
run = subprocess.run(
    [sys.executable, "-m", "scripts.check_hf_api", "--endpoint", ENDPOINT],
    cwd=PROJECT_DIR, env=env, capture_output=True, text=True,
)
print(run.stdout[-4000:])
if run.returncode not in (0, 1):
    print(run.stderr[-4000:])
print("Máy chủ đã nhận:", _get("/stats"))

archive = shutil.make_archive("/content/self_hosted_api_check", "zip", result_dir)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass
print("Đã tạo", archive)

## [Kiểm tra, tùy chọn] Gọi máy chủ Hugging Face thật (cần tài khoản còn credit)

Bài báo (mục 5 và 8) gọi FinBERT qua Hugging Face Inference API. Các ô trên chạy FinBERT ngay trong Colab (cùng model `ProsusAI/finbert`, cùng phiên bản `0bc2ac10`), nên không cần token. Ô dưới kiểm tra riêng đường gọi Hugging Face:

- Gửi 7 tiêu đề tham chiếu qua đúng adapter của backend: `top_k = 3`, softmax, gửi theo lô tối đa 15 tiêu đề, thử lại khi gặp 429/503.
- So 3 xác suất Hugging Face trả về với kết quả FinBERT local đã lưu, đếm số request HTTP.
- Ghi `hf_api_check.json` (kể cả khi lỗi, kèm mã lỗi như 402 hết credit), nén và tải về `hf_api_check.zip`.

Từ tháng 10/2026 tài khoản Hugging Face miễn phí không còn credit, nên ô này chỉ chạy được với tài khoản PRO hoặc đã nạp credit. Tài khoản miễn phí sẽ nhận HTTP 402: đó là giới hạn của Hugging Face, không phải lỗi code. Không có credit thì dùng ô máy chủ tự dựng ở trên.

Chuẩn bị: tạo token tại huggingface.co/settings/tokens (Fine-grained, tick **Make calls to Inference Providers**). Trong Colab, mở biểu tượng chìa khóa **Secrets**, thêm khóa `HF_TOKEN`, dán token vào ô giá trị và bật **Notebook access**. Ô dưới chỉ đọc Secret, không in token.

In [ ]:
# [KIỂM TRA] Gọi Hugging Face Inference API thật, nén kết quả và tải về
import os, shutil, subprocess, sys

if not globals().get("PROJECT_DIR"):
    raise RuntimeError("Hãy chạy ô '[NÂNG CẤP C] Nạp dự án và phần C' trước.")
try:
    from google.colab import userdata as _colab_secrets
    _hf_token = _colab_secrets.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")
if not _hf_token:
    print("Chưa đọc được HF_TOKEN từ Secrets: script sẽ báo lỗi 424 và hướng dẫn cách thêm.")
env = dict(os.environ)
if _hf_token:
    env["HF_TOKEN"] = _hf_token
run = subprocess.run([sys.executable, "-m", "scripts.check_hf_api"], cwd=PROJECT_DIR, env=env, capture_output=True, text=True)
print(run.stdout[-4000:])
result_dir = os.path.join(PROJECT_DIR, "artifacts", "hf_api_check")
if run.returncode not in (0, 1) or not os.path.isdir(result_dir):
    print(run.stderr[-4000:])
else:
    archive = shutil.make_archive("/content/hf_api_check", "zip", result_dir)
    try:
        from google.colab import files
        files.download(archive)
    except Exception:
        pass
    print("Đã tạo", archive)